In [ ]:
using NeuralParam
using Plots, CairoMakie

In [ ]:
TRUNC = 31
NLAYERS = 8

SG = SpectralGrid(trunc=TRUNC, nlayers=NLAYERS)

In [ ]:
rollouts = [
    "none_skill.jld2",
    "cllw_default_skill.jld2",
    "nllw_default_skill.jld2",
]

rollouts_data = []

rollouts_dir = joinpath(@__DIR__, "..", "results", "rollouts")

for r in rollouts
    data = jldload(joinpath(rollouts_dir, r))
    push!(rollouts_data, data)
end




rollouts = collect_rollouts(["none_skill", "cllw_default_skill", "nllw_default_skill"])
rollouts.cllw_default_skill        # access by name


rollouts = collect_rollouts(["cllw_default_skill", "nllw_default_skill"])
results  = merge(values(rollouts)...)     # flatten to (; cllw=…, nllw=…)
plot_rollout(; results, metric = :bias)

In [ ]:
p_rmse = plot_forecast(;rollouts_data, metric=:rmse)
p_bias = plot_forecast(;rollouts_data, metric=:bias)

In [ ]:
heatmap_days = [1,3,7,14]
layers = [NLAYERS]

titles = [String(k) for k in keys(rollouts_data)]

for layer in layers 

    # Shared colorbar calculated from first results scheme (target mostly)
    crange = extrema(reduce(vcat, (vec(f[:, layer]) for f in first(rollouts_data).heatmaps)))

    # Create and save heatmaps
    for (j, d) in enumerate(heatmap_days)
        fields = [r.heatmaps[j][:, layer] for r in rollouts_data]
        fig = plot_heatmaps(fields; titles = titles, colorrange = crange)
        CairoMakie.save(joinpath(hm_dir, "day$(d)_layer$(layer).png"), fig)
    end
end